# Fine-tune a YOLOv8 license-plate detector

Run in Google Colab with **Runtime > Change runtime type > T4 GPU** (or another available GPU), then run the cells in order. Free GPUs are not guaranteed. This notebook fine-tunes COCO-pretrained YOLOv8n, it does not train from random initialization. No training results are bundled or claimed.

## Sources and licenses
- Dataset: [keremberke/license-plate-object-detection](https://huggingface.co/datasets/keremberke/license-plate-object-detection), originally [Augmented Startups, Vehicle Registration Plates](https://universe.roboflow.com/augmented-startups/vehicle-registration-plates-trudk/dataset/1), **CC BY 4.0**. Dataset images and annotations are converted from COCO to YOLO format below. Preserve attribution and [license](https://creativecommons.org/licenses/by/4.0/) when sharing derived data.
- [Ultralytics YOLO](https://www.ultralytics.com/license): AGPL-3.0 or a separate enterprise license. Review the terms before distributing or deploying a model/project.
- This detector localizes plates. It does not read their text, and it does not establish Egyptian plate OCR support.

The converter was checked on the source's three-image validation sample. Full GPU training and this Colab environment have not been run as part of notebook preparation. Download the final weights before closing the session, since temporary Colab storage can disappear.


In [ ]:
%pip install -q ultralytics==8.4.218 huggingface_hub==0.36.0 pyyaml


In [ ]:
import torch
print('PyTorch:', torch.__version__)
assert torch.cuda.is_available(), 'No GPU assigned. Select a GPU runtime, or retry when free GPUs are available.'
print('GPU:', torch.cuda.get_device_name(0))


## Download the original train/validation/test splits

About 8,823 images in total: 6,176 training, 1,765 validation and 882 test. Do not train on the validation or test splits. The dataset revision is pinned for reproducibility. No Roboflow API key is needed for this mirror.


In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path
import zipfile

REVISION = 'a51194c739991abb50ac8afe14704aa99a66cf51'
RAW = Path('/content/plate_data_raw')
for split in ['train', 'valid', 'test']:
    archive = hf_hub_download(
        repo_id='keremberke/license-plate-object-detection',
        repo_type='dataset', filename=f'data/{split}.zip', revision=REVISION)
    with zipfile.ZipFile(archive) as z:
        base = (RAW / split).resolve()
        assert all((base / member.filename).resolve().is_relative_to(base) for member in z.infolist())
        z.extractall(base)
print('Downloaded source splits')


In [ ]:
import json
import shutil
from pathlib import Path

def convert_coco(source, target, split):
    source, target = Path(source), Path(target)
    annotation_file = next(source.rglob('_annotations.coco.json'))
    coco = json.loads(annotation_file.read_text())
    categories = {c['id']: c['name'] for c in coco['categories']}
    assert set(categories.values()) == {'license_plate'}, categories
    images = {im['id']: im for im in coco['images']}
    annotations = {key: [] for key in images}
    for ann in coco['annotations']:
        if ann.get('iscrowd', 0) or ann['category_id'] not in categories:
            continue
        im = images[ann['image_id']]
        x, y, w, h = ann['bbox']
        x1, y1 = max(0, x), max(0, y)
        x2, y2 = min(im['width'], x+w), min(im['height'], y+h)
        if x2 <= x1 or y2 <= y1:
            continue
        annotations[ann['image_id']].append(
            f"0 {(x1+x2)/2/im['width']:.8f} {(y1+y2)/2/im['height']:.8f} "
            f"{(x2-x1)/im['width']:.8f} {(y2-y1)/im['height']:.8f}")
    for directory in ['images', 'labels']:
        (target / directory / split).mkdir(parents=True, exist_ok=True)
    for image_id, im in images.items():
        src = annotation_file.parent / im['file_name']
        assert src.is_file(), src
        filename = f"{image_id}_{Path(im['file_name']).name}"
        shutil.copy2(src, target / 'images' / split / filename)
        label = target / 'labels' / split / Path(filename).with_suffix('.txt')
        label.write_text('\n'.join(annotations[image_id]) + ('\n' if annotations[image_id] else ''))
    return len(images)


In [ ]:
import yaml
DATA = Path('/content/plate_data_yolo')
expected = {'train': 6176, 'valid': 1765, 'test': 882}
for split, expected_count in expected.items():
    count = convert_coco(RAW / split, DATA, split)
    assert count == expected_count, (split, count, expected_count)
    print(split, count)
config = {'path': str(DATA), 'train': 'images/train', 'val': 'images/valid',
          'test': 'images/test', 'names': {0: 'license_plate'}}
(DATA / 'data.yaml').write_text(yaml.safe_dump(config))
print((DATA / 'data.yaml').read_text())


## Fine-tune

Start with 30 epochs, 640-pixel input and batch 8. This is a starting configuration, not an accuracy or runtime promise. If GPU memory runs out, restart the training cell with batch 4. Results and checkpoints are in temporary session storage. For longer runs, download `last.pt` regularly or save checkpoints to your own Drive explicitly.


In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
results = model.train(data=str(DATA / 'data.yaml'), epochs=30, imgsz=640,
                      batch=8, device=0, workers=2, seed=42,
                      project='/content/plate_runs', name='yolov8n', exist_ok=True)
BEST = Path(model.trainer.best)
assert BEST.is_file(), BEST
print('Best checkpoint:', BEST)


## Evaluate on the held-out test split

Only report the metrics actually printed by this run. Test metrics measure detection on this dataset, not OCR accuracy, real-time speed, or performance on Egyptian plates.


In [ ]:
best_model = YOLO(str(BEST))
metrics = best_model.val(data=str(DATA / 'data.yaml'), split='test', device=0)
print(metrics.results_dict)


## Download weights and provenance

Move `license_plate_detector.pt` into the project root. Get your own video or the separately licensed [Pexels clip](https://www.pexels.com/video/moving-cars-on-the-road-5344610/) by George Morina ([terms](https://www.pexels.com/license/)). The current project OCR accepts only seven-character LLNNLLL plates. Our test on that clip detected plates but accepted no plate text, so it is not an OCR success example.

Run `python main.py --video sample.mp4`. Keep newly generated CSVs separate from the included historical sample. Do not represent another model author's metrics as your own.


In [ ]:
import shutil, json
from google.colab import files
output = Path('/content/license_plate_detector.pt')
shutil.copy2(BEST, output)
provenance = {
    'dataset': 'keremberke/license-plate-object-detection',
    'dataset_revision': REVISION,
    'dataset_original_author': 'Augmented Startups',
    'dataset_license': 'CC BY 4.0',
    'dataset_changes': 'COCO boxes converted to YOLO normalized labels',
    'base_model': 'yolov8n.pt',
    'framework': 'ultralytics 8.4.218',
    'epochs_requested': 30,
    'test_metrics': {str(k): float(v) for k, v in metrics.results_dict.items()}
}
Path('/content/training_provenance.json').write_text(json.dumps(provenance, indent=2))
files.download(str(output))
files.download('/content/training_provenance.json')
